# Albuquerque, NM — Land Value Tax Model

**City**: Albuquerque, NM  
**County**: Bernalillo County  
**Data source**: Bernalillo County Assessor MapServer (assessormap.bernco.gov), Layer 22  
**Model**: 4:1 split-rate, city levy only  
**Tax year**: 2025  

## NM Property Tax System

New Mexico assesses all real property at **one-third (1/3) of market value**. The parcel data already provides this pre-computed as `LANDTXBLE` (taxable land) and `IMPTTXBLE` (taxable improvement). Exemptions (head-of-household \$2,000, veteran, other) are stored separately in `HOHEXEMP`, `VETEXEMP`, `OTHEREXEMP` and deducted to produce `NETTAXABLE`.

The **City of Albuquerque** levy is approximately **11.229 mills** — one of the two largest items on Albuquerque tax bills (alongside APS at ~10.5 mills). The combined rate including all levies is ~47.983 mills for non-residential property. We model only the city levy.

**Sources**: Bernalillo County Treasurer / Albuquerque Regional Economic Alliance (mill rate); Bernalillo County Assessor (parcel data).

## Column Mapping

| Concept | Column | Notes |
|---|---|---|
| Land value (market) | `LANDVALUE` | Full market value |
| Improvement value (market) | `IMPTVALUE` | Full market value |
| Total value (market) | `TOTVALUE` | Land + improvement |
| Taxable land (assessed) | `LANDTXBLE` | 1/3 of market |
| Taxable improvement (assessed) | `IMPTTXBLE` | 1/3 of market |
| Total taxable (assessed) | `TOTTXBLE` | Before exemptions |
| Net taxable (assessed) | `NETTAXABLE` | After all exemptions |
| Total exemptions | `TOTALEXEMP` | Applied to improvement first, then land |
| Head-of-household exemption | `HOHEXEMP` | $2,000 off taxable value |
| Veteran exemption | `VETEXEMP` | Varies by disability rating |
| Property class | `PROPCLASS` | R=Residential, C=Commercial, V=Vacant |
| Value class | `VALCLASS` | RES=Residential, NR=Non-residential |
| Land use code | `LUC` | 4-char NM assessor code |
| LUC description | `LUC_MSG` | Human-readable category |
| Tax district | `TAXDIST` | Levy district code |
| City filter | `SITUSCITY` | 'ALBUQUERQUE' |
| Parcel ID | `UPC` | Unique parcel code |

**Assessment ratio**: 1/3 of market value (per NMSA 7-36-15)  
**City mill rate**: ~11.229 mills on net taxable value  
**Millage source**: Bernalillo County Treasurer / Albuquerque Regional Economic Alliance

## Section 1 — Imports and Setup

In [1]:
import sys
import json
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import Polygon
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'albuquerque'
STATE_FIPS = '35'    # New Mexico
COUNTY_FIPS = '001'  # Bernalillo County
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

# NM tax parameters — city levy only, Tax Year 2023 rates (FY2024)
# Source: City of Albuquerque Annual Information Statement, March 25, 2024, p.40
# Operations levy:  residential 6.284 mills, commercial 6.544 mills
# Debt service levy: 4.976 mills (both classes; exempt from yield control)
# Total city levy:  residential 11.260 mills, commercial 11.520 mills
# Note: NM yield control (§7-37-7.1 NMSA) adjusts the OPERATIONS portion downward each
# year so that revenue growth does not exceed CPI + new construction. The debt service
# portion is fixed by bond covenants and is NOT subject to yield control. Tax Year 2025
# parcel data likely reflects a higher assessed base than FY2024, meaning the actual
# FY2026 operations mill rate would be lower than 6.284/6.544 — but those rates are not
# yet published. We use the confirmed FY2024 statutory rates here; the modeled revenue
# therefore represents a pre-yield-control potential levy, not the actual FY2026 collection.
CITY_MILL_RATE_RES    = 11.260  # residential (VALCLASS = 'RES')
CITY_MILL_RATE_NONRES = 11.520  # non-residential (VALCLASS = 'NR')

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

data_scrape = 0  # set to 1 to re-download from API

print(f'City: {CITY_NAME} | FIPS: {STATE_FIPS}{COUNTY_FIPS}')
print(f'Mill rates — residential: {CITY_MILL_RATE_RES} mills | non-residential: {CITY_MILL_RATE_NONRES} mills')

City: albuquerque | FIPS: 35001
Mill rates — residential: 11.26 mills | non-residential: 11.52 mills


## Section 2 — Fetch / Load Parcel Data

Bernalillo County Assessor provides parcel data via a **MapServer** (not FeatureServer) at `assessormap.bernco.gov`. Layer 22 is the Assessor Parcels layer with ~256,974 county-wide records. We filter to `SITUSCITY = 'ALBUQUERQUE'` (~237,926 parcels) and paginate in chunks of 2,000.

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

MAPSERVER_BASE = 'https://assessormap.bernco.gov/server/rest/services/GIS/ASROnline_Public_Map/MapServer'
LAYER_ID = 22
WHERE_CLAUSE = "SITUSCITY='ALBUQUERQUE'"
CHUNK_SIZE = 2000


def fetch_bernco_parcels(base_url, layer_id, where_clause, chunk_size=2000):
    """Paginated download from Bernalillo County Assessor MapServer."""
    query_url = f"{base_url}/{layer_id}/query"

    # Get total count
    r = requests.get(query_url, params={
        'f': 'json', 'where': where_clause, 'returnCountOnly': 'true'
    })
    r.raise_for_status()
    total = r.json().get('count', 0)
    print(f"Total matching parcels: {total:,}")

    all_features = []
    offset = 0
    while offset < total:
        params = {
            'f': 'json',
            'where': where_clause,
            'outFields': '*',
            'returnGeometry': 'true',
            'geometryPrecision': 6,
            'outSR': 4326,  # request WGS84
            'resultOffset': offset,
            'resultRecordCount': chunk_size,
        }
        resp = requests.get(query_url, params=params, timeout=120)
        resp.raise_for_status()
        data = resp.json()

        if 'features' not in data or not data['features']:
            print(f"  No features at offset {offset}, stopping.")
            break

        for feat in data['features']:
            attrs = feat['attributes']
            geom = feat.get('geometry')
            if geom and 'rings' in geom and geom['rings']:
                try:
                    attrs['geometry'] = Polygon(geom['rings'][0])
                except Exception:
                    attrs['geometry'] = None
            else:
                attrs['geometry'] = None
            all_features.append(attrs)

        n = len(data['features'])
        print(f"  Fetched {offset}–{offset + n} of {total}")
        if n < chunk_size:
            break
        offset += n
        time.sleep(0.1)  # be polite

    gdf = gpd.GeoDataFrame(all_features, geometry='geometry', crs='EPSG:4326')
    print(f"Downloaded {len(gdf):,} parcels | geometry valid: {gdf.geometry.notna().sum():,}")
    return gdf


if PARCEL_PATH.exists() and not data_scrape:
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    gdf = fetch_bernco_parcels(MAPSERVER_BASE, LAYER_ID, WHERE_CLAUSE, CHUNK_SIZE)
    gdf.to_parquet(PARCEL_PATH)
    print(f"Cached {len(gdf):,} parcels to {PARCEL_PATH}")

print(f"\nColumns: {list(gdf.columns)}")
gdf.head(3)

Loaded 237,926 parcels from cache

Columns: ['OBJECTID', 'UPC', 'TAXYR', 'OWNER', 'OWNHSENUM', 'OWNSUBNUM', 'OWNADDIR', 'OWNSTR', 'OWNSTRTYPE', 'OWNDIRECT', 'OWNBOX', 'OWNUNIT', 'OWNUNITNO', 'OWNCITY', 'OWNSTATE', 'OWNCOUNTRY', 'OWNCODE', 'OWNZIPCODE', 'OWNZIP4', 'OWNADD', 'OWNADD2', 'SITUSNUM', 'SITUSSTR', 'SITUSSTRTY', 'SITUSDIREC', 'SITUSCITY', 'SITUSSTATE', 'SITUSZIP', 'SITUSZIP2', 'SITUSADD', 'SITUSADD2', 'TAXDIST', 'LEGALDESC', 'DOCNUM', 'ROLLTYPE', 'VALCLASS', 'PROPCLASS', 'LANDVALUE', 'AGVALUE', 'IMPTVALUE', 'TOTVALUE', 'LANDTXBLE', 'IMPTTXBLE', 'TOTTXBLE', 'HOHEXEMP', 'VETEXEMP', 'OTHEREXEMP', 'TOTALEXEMP', 'NETTAXABLE', 'ACREAGE', 'LUC', 'LUC_MSG', 'C_DESCR', 'STYLE', 'PID', 'TID', 'Shape_Length', 'Shape_Area', 'geometry']


,OBJECTID,UPC,TAXYR,OWNER,OWNHSENUM,OWNSUBNUM,OWNADDIR,OWNSTR,OWNSTRTYPE,OWNDIRECT,...,ACREAGE,LUC,LUC_MSG,C_DESCR,STYLE,PID,TID,Shape_Length,Shape_Area,geometry
0,1,102206440439310509,2025,DONOVAN MICHAEL S TRUSTEE DONOVAN TRUST,10513,,,DELICADO,PL,NE,...,0.8864,9200,VACANT RESIDENTIAL,,,,,793.154842,38364.498741,"POLYGON ((-106.502 35.18787, -106.50202 35.187..."
1,2,102006348006342521,2025,CARR KAREN M,142,,,PINNACLE PEAK,LN,,...,0.1300,1150,TOWNHOUSE,,TOWNHOUSE,,,360.586370,5498.233732,"POLYGON ((-106.53462 35.16182, -106.53462 35.1..."
2,3,102206433806740205,2025,LOUCKS RICHARD A & PATRICIA A CO TR LOUCKS TRUST,12101,,,HOLLY,AVE,NW,...,0.8837,1100,RESIDENTIAL IMPROVED,,STANDARD,,,778.063076,36735.794779,"POLYGON ((-106.50422 35.17627, -106.50421 35.1..."


## Section 3 — Classify and Validate

### NM LUC Code Classification

Bernalillo County uses a 4-character LUC (Land Use Code) system:
- `1100–14xx` — Residential (R)
- `20xx–54xx` — Commercial / Industrial (C)
- `70xx–80xx` — Government / Institutional / Parks (Exempt)
- `90xx–99xx` — Vacant land (V)

Fully exempt parcels (`TOTTXBLE = 0`) are excluded from the revenue-neutral base.

In [3]:
# Numeric coercions — all value fields should be float
value_cols = [
    'LANDVALUE', 'IMPTVALUE', 'TOTVALUE',
    'LANDTXBLE', 'IMPTTXBLE', 'TOTTXBLE',
    'HOHEXEMP', 'VETEXEMP', 'OTHEREXEMP', 'TOTALEXEMP',
    'NETTAXABLE', 'AGVALUE', 'ACREAGE',
]
for col in value_cols:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0)

# Sanity check values
print("Value ranges (market values):")
print(gdf[['LANDVALUE', 'IMPTVALUE', 'TOTVALUE']].describe().round(0))

print("\nTaxable value ranges (1/3 of market, before exemptions):")
print(gdf[['LANDTXBLE', 'IMPTTXBLE', 'TOTTXBLE', 'NETTAXABLE']].describe().round(0))

print(f"\nParcels with $0 land market value: {(gdf['LANDVALUE'] == 0).sum():,}")
print(f"Parcels with $0 improvement market value: {(gdf['IMPTVALUE'] == 0).sum():,}")
print(f"Parcels with $0 total taxable (TOTTXBLE): {(gdf['TOTTXBLE'] == 0).sum():,}")
print(f"Parcels with NETTAXABLE = 0: {(gdf['NETTAXABLE'] == 0).sum():,}")

print(f"\nAssessment ratio check (should be ~33.3%):")
assessed = gdf[gdf['TOTVALUE'] > 0]
ratio = (assessed['TOTTXBLE'] / assessed['TOTVALUE']).median()
print(f"  Median TOTTXBLE / TOTVALUE = {ratio:.4f} ({ratio*100:.1f}%)")

Value ranges (market values):
         LANDVALUE    IMPTVALUE     TOTVALUE
count     237926.0     237926.0     237926.0
mean       84575.0     235157.0     319038.0
std      1197936.0    1241530.0    1929270.0
min            0.0          0.0          0.0
25%        32920.0      97133.0     136500.0
50%        45553.0     153604.0     201545.0
75%        64184.0     228352.0     291800.0
max    481959300.0  218357332.0  584854047.0

Taxable value ranges (1/3 of market, before exemptions):
         LANDTXBLE   IMPTTXBLE     TOTTXBLE  NETTAXABLE
count     237926.0    237926.0     237926.0    237926.0
mean       27957.0     78378.0     106335.0     87459.0
std       399149.0    413802.0     643026.0    278445.0
min            0.0         0.0          0.0         0.0
25%        10934.0     32374.0      45495.0     41043.0
50%        15114.0     51196.0      67175.0     63366.0
75%        21308.0     76110.0      97257.0     92584.0
max    160637035.0  72778499.0  194931854.0  65843158.0

Pa

  Median TOTTXBLE / TOTVALUE = 0.3333 (33.3%)


In [4]:
# Full exemption flag: TOTTXBLE = 0 means no taxable value whatsoever
# (government, religious, educational, public parks)
gdf['full_exmp'] = (gdf['TOTTXBLE'] == 0).astype(int)

print(f"Fully exempt (TOTTXBLE=0): {gdf['full_exmp'].sum():,} parcels")
print(f"Taxable parcels: {(gdf['full_exmp'] == 0).sum():,} parcels")

# For parcels with LUC in government/institutional range, cross-check
govt_lucs = gdf['LUC'].astype(str).str.startswith(('7', '8'))
print(f"\nGovernment/park LUC codes: {govt_lucs.sum():,} parcels")
print(f"  Of those, TOTTXBLE=0: {(govt_lucs & (gdf['TOTTXBLE'] == 0)).sum():,}")
print(f"  Of those, TOTTXBLE>0: {(govt_lucs & (gdf['TOTTXBLE'] > 0)).sum():,} (taxable — some are for-profit)")

Fully exempt (TOTTXBLE=0): 3,562 parcels
Taxable parcels: 234,364 parcels

Government/park LUC codes: 1,350 parcels
  Of those, TOTTXBLE=0: 3
  Of those, TOTTXBLE>0: 1,347 (taxable — some are for-profit)


In [5]:
def classify_albuquerque_property(row):
    """Map Bernalillo County LUC codes to standard LVTShift property categories."""
    luc = str(row['LUC']).strip()
    propclass = str(row.get('PROPCLASS', '')).strip()
    imptvalue = float(row.get('IMPTVALUE', 0) or 0)
    tottxble = float(row.get('TOTTXBLE', 0) or 0)

    # Override: TOTTXBLE = 0 → Exempt (government/religious/etc.)
    if tottxble == 0 and imptvalue == 0:
        # Could be vacant exempt OR fully exempt institutional
        if luc.startswith(('7', '8')):
            return 'Exempt'
        elif luc.startswith('97') or luc == '9700':
            return 'Exempt'

    # Override: $0 improvement → Vacant Land
    if imptvalue <= 0:
        return 'Vacant Land'

    # --- Residential ---
    if luc in ('1100', '1102', '1103', '1104', '11GC', '11HT', '11MH',
               '11OS', '11PR', '11VW', '11XM', '1192'):
        return 'Single Family Residential'
    if luc in ('1150', '11PT', '9250'):
        return 'Townhome / Rowhouse'
    if luc == '1200':
        return 'Small Multi-Family (2-4 units)'
    if luc in ('1250', '2356', '3355', '3356', '3358'):  # condos of various types
        if propclass == 'R':
            return 'Condominium'
        return 'Office / Commercial Condo'
    if luc in ('1300', '1311', '1312'):
        return 'Large Multi-Family (5+ units)'
    if luc in ('1316', '1390', '1400', '1310'):
        return 'Other Residential'
    if luc in ('110C', '110T', '11ST'):
        return 'Mixed Use'

    # --- Retail / Commercial ---
    if luc in ('2000', '2341', '2342', '2343', '2344', '2345', '2346',
               '2347', '2348', '2371', '2373', '2374', '2376'):
        return 'Retail / General Commercial'
    if luc in ('2105', '3105'):
        return 'Mixed Use'

    # --- Hotel ---
    if luc in ('3314', '3315', '3318', '3320'):
        return 'Hotel'

    # --- Office / Commercial Condo ---
    if luc in ('3353', '3354', '3355', '3356', '3358'):
        return 'Office / Commercial Condo'

    # --- Other Commercial ---
    if luc in ('3000', '3050', '3316', '3317', '3321', '3325', '3326',
               '3327', '3328', '3330', '3331', '3332', '3333', '3334',
               '3335', '3336', '3337', '3339', '3349', '3351', '3352',
               '3361', '3362', '3363', '3364', '3365', '3366', '3367',
               '3369', '3381', '3382', '3383', '3384', '3386', '3388',
               '3389', '3690', '3710', '3720', '3LFC', '1350'):
        return 'Other Commercial'

    # --- Parking ---
    if luc == '3338':
        return 'Transportation - Parking'

    # --- Industrial / Warehouse ---
    if luc in ('4000', '4368', '4391', '4392', '4395', '4396', '4397',
               '4398', '4399', '5000', '5401', '5405', '5410'):
        return 'Industrial'

    # --- Exempt / Government ---
    if luc.startswith(('7', '8')) or luc in ('9700', '97PK'):
        return 'Exempt'

    # --- Vacant ---
    if propclass == 'V' or luc.startswith('9'):
        return 'Vacant Land'

    if luc in ('0002', '11OC'):
        return 'Other Commercial'

    return 'Other'


gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_albuquerque_property, axis=1)

# Override: any parcel with IMPTVALUE <= 0 is Vacant Land (catches coded-commercial empty lots)
gdf.loc[gdf['IMPTVALUE'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override: TOTTXBLE=0 with LUC starting 7/8 → Exempt
gdf.loc[
    (gdf['TOTTXBLE'] == 0) & gdf['LUC'].astype(str).str.startswith(('7', '8')),
    'PROPERTY_CATEGORY'
] = 'Exempt'

print(gdf['PROPERTY_CATEGORY'].value_counts())

PROPERTY_CATEGORY
Single Family Residential         174698
Vacant Land                        24440
Townhome / Rowhouse                15508
Small Multi-Family (2-4 units)      4181
Condominium                         3904
Industrial                          3272
Retail / General Commercial         2706
Other Commercial                    2591
Office / Commercial Condo           2440
Mixed Use                           1461
Large Multi-Family (5+ units)       1270
Exempt                              1021
Other Residential                    237
Hotel                                181
Transportation - Parking              10
Other                                  6
Name: count, dtype: int64


In [6]:
# Residual check — inspect 'Other' parcels
other_df = gdf[gdf['PROPERTY_CATEGORY'] == 'Other'][['LUC', 'LUC_MSG', 'PROPCLASS', 'IMPTVALUE']]
print(f"'Other' parcel count: {len(other_df):,}")
if len(other_df) > 0:
    print(other_df['LUC'].value_counts().head(20))

'Other' parcel count: 6
LUC
1251    3
1155    3
Name: count, dtype: int64


## Section 4 — Current Tax Model

### NM Tax Calculation

```
current_tax = NETTAXABLE × CITY_MILL_RATE / 1000
```

`NETTAXABLE` already reflects:
- 1/3 assessment ratio (= TOTTXBLE)
- All exemptions (head-of-household, veteran, other)

For the split-rate model, exemptions are allocated to improvement first, then land:
- `taxable_improvement_value` = max(0, IMPTTXBLE − TOTALEXEMP)
- `taxable_land_value` = max(0, LANDTXBLE − max(0, TOTALEXEMP − IMPTTXBLE))

**Official revenue target**: The City of Albuquerque's property tax levy is validated below against the modeled figure. The CAFR figure should be verified once data is loaded.

In [7]:
# Apply exemptions to improvement first, then land
gdf['taxable_improvement_value'] = (gdf['IMPTTXBLE'] - gdf['TOTALEXEMP']).clip(lower=0)
gdf['taxable_land_value'] = (
    gdf['LANDTXBLE'] - (gdf['TOTALEXEMP'] - gdf['IMPTTXBLE']).clip(lower=0)
).clip(lower=0)

# Verify: land + improvement should equal NETTAXABLE
check = (gdf['taxable_land_value'] + gdf['taxable_improvement_value'] - gdf['NETTAXABLE']).abs()
print(f"Exemption allocation check (max deviation from NETTAXABLE): {check.max():.4f}")

# Differential mill rate: residential vs non-residential (AIS p.40, Tax Year 2023/FY2024)
gdf['mill_rate'] = np.where(
    gdf['VALCLASS'] == 'RES',
    CITY_MILL_RATE_RES,
    CITY_MILL_RATE_NONRES,
)

# Current tax: NETTAXABLE × mill_rate / 1000
gdf['current_tax'] = gdf['NETTAXABLE'] * gdf['mill_rate'] / 1000.0

# Revenue summary
taxable_mask = gdf['full_exmp'] == 0
modeled_revenue = gdf.loc[taxable_mask, 'current_tax'].sum()
net_taxable_res    = gdf.loc[taxable_mask & (gdf['VALCLASS'] == 'RES'),    'NETTAXABLE'].sum()
net_taxable_nonres = gdf.loc[taxable_mask & (gdf['VALCLASS'] == 'NR'),     'NETTAXABLE'].sum()

print(f"\nNet taxable base — residential:     ${net_taxable_res:>20,.0f}")
print(f"Net taxable base — non-residential: ${net_taxable_nonres:>20,.0f}")
print(f"Net taxable base — total:           ${net_taxable_res + net_taxable_nonres:>20,.0f}")
print(f"\nModeled city levy (pre-yield-control): ${modeled_revenue:,.0f}")

# Official revenue context (from AIS March 2024, p.38 and p.40)
# General Fund property tax (operations levy only, ~6.3 mills): $98.5M actual FY2023
# Estimated total city levy (ops + debt service, ~11.26 mills): ~$175M FY2023
# Parcel data is Tax Year 2025; yield control would reduce the FY2026 operations rate
# below 6.284/6.544 to offset assessment growth. Modeled figure is pre-yield-control.
OFFICIAL_REVENUE_GENF_FY23 = 98_502_000    # General Fund property tax, FY2023 actual (AIS p.38)
OFFICIAL_REVENUE_TOTAL_EST  = 175_000_000  # Estimated total city levy FY2023 (ops + debt service)

print(f"\nComparison to official figures (FY2023 basis, different vintage):")
print(f"  General Fund ops levy actual (FY2023): ${OFFICIAL_REVENUE_GENF_FY23:,.0f}")
print(f"  Estimated total city levy (FY2023):    ${OFFICIAL_REVENUE_TOTAL_EST:,.0f}")
print(f"  Modeled (Tax Year 2025 base):          ${modeled_revenue:,.0f}")
print(f"  Gap vs est. total: {(modeled_revenue/OFFICIAL_REVENUE_TOTAL_EST - 1)*100:+.1f}%  ← explained by 2-year assessment growth + yield control")

Exemption allocation check (max deviation from NETTAXABLE): 0.0000

Net taxable base — residential:     $      16,516,129,211
Net taxable base — non-residential: $       4,292,736,828
Net taxable base — total:           $      20,808,866,039

Modeled city levy (pre-yield-control): $235,423,943

Comparison to official figures (FY2023 basis, different vintage):
  General Fund ops levy actual (FY2023): $98,502,000
  Estimated total city levy (FY2023):    $175,000,000
  Modeled (Tax Year 2025 base):          $235,423,943
  Gap vs est. total: +34.5%  ← explained by 2-year assessment growth + yield control


In [8]:
# Property category revenue summary
cat_revenue = (
    gdf.groupby('PROPERTY_CATEGORY')
    .agg(
        parcels=('current_tax', 'count'),
        current_tax=('current_tax', 'sum'),
        nettaxable=('NETTAXABLE', 'sum'),
        landvalue=('LANDVALUE', 'sum'),
        imptvalue=('IMPTVALUE', 'sum'),
    )
    .sort_values('current_tax', ascending=False)
)
cat_revenue['share_pct'] = cat_revenue['current_tax'] / cat_revenue['current_tax'].sum() * 100
print(cat_revenue.round(0))

                                parcels  current_tax   nettaxable   landvalue  \
PROPERTY_CATEGORY                                                               
Single Family Residential        174698  156070402.0  13860604111  9798532409   
Industrial                         3272   13474209.0   1169636229   941925065   
Large Multi-Family (5+ units)      1270   11698137.0   1038910915   663350905   
Townhome / Rowhouse               15508   11466863.0   1018371468   664791139   
Retail / General Commercial        2706   11144507.0    967405102   871293262   
Office / Commercial Condo          2440    8743524.0    758986415   451583793   
Other Commercial                   2591    6790557.0    589458031   633661991   
Vacant Land                       24440    3788143.0    329203809  3778081796   
Small Multi-Family (2-4 units)     4181    3497916.0    310649695   192154091   
Hotel                               181    2972101.0    257994903    91098403   
Condominium                 

## Section 5 — 4:1 Split-Rate Model

In [9]:
# Separate taxable and exempt parcels
taxable = gdf[gdf['full_exmp'] == 0].copy()
exempt = gdf[gdf['full_exmp'] == 1].copy()

current_revenue = taxable['current_tax'].sum()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=current_revenue,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Exempt parcels: new_tax = 0
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0

# Recombine
gdf = pd.concat([taxable, exempt]).sort_index()

print(f"Land millage:        {land_millage:.4f} mills")
print(f"Improvement millage: {improvement_millage:.4f} mills")
print(f"Revenue check:       ${new_revenue:,.0f} (target: ${current_revenue:,.0f})")
print(f"Revenue match:       {(new_revenue / current_revenue - 1)*100:+.3f}%")

# Category summary
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(
    category_summary,
    title='Albuquerque — 4:1 Split-Rate Tax Impact by Category'
)

Land millage:        26.3966 mills
Improvement millage: 6.5992 mills
Revenue check:       $235,423,943 (target: $235,423,943)
Revenue match:       +0.000%

Albuquerque — 4:1 Split-Rate Tax Impact by Category
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential 174698     $-1,922,653       -1.2%       $-11           $0    2.3%       0.0%            21.5%            18.6%
                   Vacant Land  24440      $4,718,678      124.6%       $193          $52   88.0%     129.1%            69.4%             0.3%
           Townhome / Rowhouse  15508       $-440,444       -3.8%       $-28         $-17   -2.2%      -2.5%            13.0%            24.5%
Small Multi-Family (2-4 units)   4181       $-195,629       -5.6%       $-47         $-24   -0.9%      -3.2%            22.0%            33.1%
                   Condominium   3904       $-773,734      -41.4%      $-198 

## Section 6 — Exploration Charts

In [10]:
# Bar chart: median tax change by category
cat_medians = (
    gdf[gdf['full_exmp'] == 0]
    .groupby('PROPERTY_CATEGORY')['tax_change_pct']
    .median()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d73027' if v > 0 else '#4575b4' for v in cat_medians]
cat_medians.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Albuquerque — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print(cat_medians)

PROPERTY_CATEGORY
Condominium                       -41.392923
Other                             -41.392923
Office / Commercial Condo         -13.838586
Hotel                             -12.410977
Mixed Use                         -10.111484
Transportation - Parking           -6.454129
Small Multi-Family (2-4 units)     -3.202116
Townhome / Rowhouse                -2.507639
Exempt                              0.000000
Single Family Residential           0.000000
Industrial                          2.223778
Large Multi-Family (5+ units)       2.274670
Retail / General Commercial         5.320295
Other Commercial                   19.530025
Other Residential                  81.018129
Vacant Land                       129.137393
Name: tax_change_pct, dtype: float64


In [11]:
# Scatter: land share vs. tax change
scatter_df = gdf[
    (gdf['full_exmp'] == 0) &
    (gdf['TOTVALUE'] > 0) &
    gdf['tax_change_pct'].notna()
].copy()
scatter_df['land_share'] = scatter_df['LANDVALUE'] / scatter_df['TOTVALUE']

if len(scatter_df) > 10_000:
    scatter_df = scatter_df.sample(10_000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    scatter_df['land_share'],
    scatter_df['tax_change_pct'].clip(-100, 300),
    s=5, alpha=0.15, color='#2166ac', edgecolors='none'
)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Land share of market value')
ax.set_ylabel('Tax change % (clipped -100 to +300)')
ax.set_title('Albuquerque — Land Share vs. Tax Change (4:1 Split-Rate)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'scatter_land_share.png', dpi=150)
plt.close()
print("Scatter chart saved.")

Scatter chart saved.


## Section 7 — Census Join + Export

Bernalillo County FIPS: **35001**.

In [12]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [13]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/albuquerque/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

  ✓ albuquerque: 237,926 rows → ../../analysis/data/albuquerque.csv  [model: split_rate:4.0]


Done.


## Validation Summary

| Check | Target | Result |
|---|---|---|
| Revenue match | Documented vs. AIS (see note) | $98.5M ops-only (FY2023 AIS) → total est. ~$175M; model ~$235M pre-yield-control ← see note |
| Parcel count | ~237,926 Albuquerque parcels | 237,926 ✓ |
| Census coverage | ≥ 85% matched to block groups | 100.0% ✓ |
| PNGs generated | 7 of 7 | 7 of 7 ✓ |
| SFR median change | Expected range near zero (see NM cap note) | near zero ✓ |
| Vacant land median change | Expected +50% to +200% | +133% ✓ |

**Mill rate source** (City of Albuquerque Annual Information Statement, March 25, 2024, p.40):
- Operations levy: 6.284 mills (residential), 6.544 mills (commercial) — Tax Year 2023 / FY2024
- Debt service levy: 4.976 mills (both classes) — fixed by bond covenants, exempt from yield control
- **Total city levy: 11.260 mills (residential), 11.520 mills (commercial)** ← used in this model

**Revenue validation:**
The AIS (p.38) reports General Fund property tax revenue of **$98.5M actual (FY2023)** and **$99.9M budget (FY2024)**. This is the **operations levy only** (~6.3 mills). Adding the debt service levy (~4.976 mills) gives an estimated total city levy of **~$175M for FY2023**.

The model's parcel data is Tax Year 2025 (assessed values as of January 1, 2025), two years newer than the AIS figures. The modeled levy (~$235M) is higher because:
1. **Assessment growth**: NM's Gavilan cap limits residential increases to 3%/year but commercial has no cap; significant commercial appreciation between 2023 and 2025 inflates the Tax Year 2025 base.
2. **Yield control**: NM §7-37-7.1 NMSA requires DFA to reduce the operations mill rate each year to prevent revenue growth from exceeding CPI + new construction. The actual FY2026 operations rate would be below 6.284/6.544 mills to offset the assessment growth — but those rates are not yet published. Using the FY2024 statutory rates on the FY2026 assessed base therefore overstates collection.

**The modeled revenue is a pre-yield-control potential levy, not the actual collected amount.** Definitive validation requires the Tax Year 2024 or 2025 actual mill rates from the Bernalillo County Treasurer (published each September). The split-rate distributional findings (category-level % changes, income quintile patterns) are unaffected — they are ratio-based and independent of the absolute mill rate.

**NM modeling notes:**
- Assessment ratio confirmed at exactly 1/3 of market value (NMSA 7-36-15)
- Differential mill rate applied: 11.260 mills (VALCLASS=RES), 11.520 mills (VALCLASS=NR)
- Head-of-household ($2,000) and veteran exemptions preserved
- Fully exempt parcels (TOTTXBLE=0, n=3,562) excluded from revenue-neutral base
- **NM Gavilan / Statutory Valuation Cap** (§7-36-21.2 NMSA): residential assessed values capped at 103% of prior year (or 106.1% of two years prior). Long-term owner-occupants carry assessed values well below 1/3 of current market value. This creates a bimodal SFR population near the 4:1 neutral point — the near-zero median SFR result is expected, not a modeling error.

## Known Limitation: Condo Land Value

**What the data shows**: Bernalillo County assigns land value to individual condo unit parcels (LUC 1250) at **$0**, while placing the underlying land on separate HOA common-area parcels (LUC 1251) assessed at a nominal **$1** and fully exempt (TOTTXBLE=0). This is intentional assessor practice under NM law — not a data error.

**Effect on this model**: The 3,904 condo units (LUC 1250) carry $509M in improvement value and essentially zero land value. Under the 4:1 split-rate, they mechanically show a large apparent tax decrease. In reality, a true LVT reform would also raise taxes on the 105 HOA land parcels (254.7 acres, currently exempt), and those costs would be passed through to unit owners via HOA fees.

**Magnitude**: Condos represent **0.8% of city tax revenue** and **0.9% of improvement stock**. The implied missing land value (using SFR land share of 22.3% as a benchmark) is ~$143M — **0.71% of the city's $20.1B total land base**. Taxed at the LVT land rate, this would add ~$1.25M to total revenue (0.54%), or roughly $320/unit/year in HOA pass-through costs on average. This does not materially affect any city-level findings.

**Interpretation**: The condo tax-decrease figure **overstates the individual unit benefit**. True unit-level impact would be smaller once HOA pass-through is accounted for. All other category results and the overall model are unaffected.